# Chemical biology landscape analysis

We’ll analyse how combinations of biological components shape an experimental response, from preparing the measurements to exploring ruggedness, interactions and navigability. A screen of two-domain CRISPR activators will serve as our example.

## 1. Importing necessary dependencies

Install GraphFLA with pip, then import the dependencies below.

In [ ]:
!pip install graphfla

In [1]:
import graphfla
from graphfla.landscape import Landscape
import pandas as pd

## 2. Loading the experimental dataset

For our example, we use the bipartite activator screen from [Giddins et al. (2025)](https://doi.org/10.1038/s41467-025-65986-4). The constructs contain two ordered domains attached to an MS2 coat-protein scaffold for CRISPR activation. Each position has 25 choices, giving 625 constructs.

We’ll focus on activation of the target gene **EPCAM**. Its score comes from barcode sequencing after cells were sorted into expression bins; larger values indicate stronger activation. The CSV contains the mean of two biological replicates. Domain order matters: exchanging the first and second domains produces a different construct.

| Column | Meaning |
|---|---|
| `ad1_name` | Domain in the first position, nearest the scaffold. |
| `ad2_name` | Domain in the second position. |
| `activation_EPCAM` | Mean EPCAM activation score; larger is better. |

The domain names identify the source protein and any tandem-copy suffix. Let’s load these three columns and look at the first few constructs.

In [2]:
data_url = (
    "https://raw.githubusercontent.com/COLA-Laboratory/GraphFLA/"
    "main/data/ChemBio/CrisprPair/bipartite.csv"
)
df = pd.read_csv(data_url)[["ad1_name", "ad2_name", "activation_EPCAM"]]
df.head()

,ad1_name,ad2_name,activation_EPCAM
0,HHV1_VP16x2,HHV1_VP16x2,195.911720
1,HHV1_VP16x2,CercAHV2_VP16x2,217.349860
2,HHV1_VP16x2,HS_MLLx3,323.928745
3,HHV1_VP16x2,JMSFV_BEL1,170.822752
4,HHV1_VP16x2,FBAHV1_VP64,258.261613


## 3. Preparing the inputs

To construct a landscape, GraphFLA only needs two inputs—much like the features `X` and targets `y` used to train a machine-learning model:

- `X`: the experimental configurations, as a DataFrame or a two-dimensional NumPy array.
- `f`: one measured outcome per configuration, as a Series, list or one-dimensional array.

The rows of `X` and `f` must refer to the same configurations in the same order.


Here, the two domain choices form `X`, and EPCAM activation forms `f`. The domain labels are categories, with no numerical ordering. We focus on activation alone; toxicity is a separate measurement in the original screen.

In [3]:
X = df[['ad1_name', 'ad2_name']]
f = df['activation_EPCAM']
X.head()

,ad1_name,ad2_name
0,HHV1_VP16x2,HHV1_VP16x2
1,HHV1_VP16x2,CercAHV2_VP16x2
2,HHV1_VP16x2,HS_MLLx3
3,HHV1_VP16x2,JMSFV_BEL1
4,HHV1_VP16x2,FBAHV1_VP64


## 4. Constructing the landscape

With the inputs ready, we can use `Landscape` and declare both domain positions as categorical. Neighbouring constructs differ in one domain choice while the other position stays fixed. This compares domain replacements, not single-amino-acid mutations.

We set `maximize=True` because stronger EPCAM activation is our objective.

In [4]:
landscape = Landscape(maximize=True)
landscape.build_from_data(
    X, f,
    data_types={"ad1_name": "categorical", "ad2_name": "categorical"},
)

Building Landscape from data...


 - Validating data types dictionary...


   - Data types dictionary validation successful.


 - Handling missing values and duplicates...


 - Checking for missing values...


...

   - Found 2 local optima.


 - Determining global optimum...


   - Global optimum found at index 482 with fitness 675.7559.


Landscape built successfully.



Landscape(kind='default', maximize=True)

## 5. Inspecting the landscape

Let’s first look at what we’ve built. Printing the landscape shows its variables, configurations, improving edges and local optima:

In [5]:
print(landscape)

Landscape(kind='default'): 2 variables, 625 configurations, 15000 edges, 2 local optima


For a closer look at individual constructs, we can call `get_data()`. The selected outcome appears as `fitness`; `out_degree` counts directly improving moves, `in_degree` counts incoming moves from lower-scoring neighbours, and `is_lo` indicates local-optimum membership.

In [6]:
landscape.get_data()

,ad1_name,ad2_name,fitness,in_degree,out_degree,is_lo
0,HHV1_VP16x2,HHV1_VP16x2,195.911720,11,37,False
1,HHV1_VP16x2,CercAHV2_VP16x2,217.349860,10,38,False
2,HHV1_VP16x2,HS_MLLx3,323.928745,31,17,False
3,HHV1_VP16x2,JMSFV_BEL1,170.822752,3,45,False
4,HHV1_VP16x2,FBAHV1_VP64,258.261613,13,35,False
...,...,...,...,...,...,...
620,EC_TRXA,EBV_RTA,264.557822,20,28,False
621,EC_TRXA,HS_HSF1,445.440542,33,15,False
622,EC_TRXA,SG_GB1,133.866131,4,44,False
623,EC_TRXA,EC_MBP,114.287051,2,46,False


We can also summarise the activation scores with pandas:

In [7]:
landscape.get_data()["fitness"].describe()

count    625.000000
mean     318.139098
std      123.523632
min      102.893841
25%      224.022698
50%      293.891485
75%      395.613920
max      675.755861
Name: fitness, dtype: float64

To inspect the construct with the highest EPCAM activation score, we can use the global-optimum node index:

In [8]:
landscape[landscape.go_index]

{'ad1_name': 'HS_P65',
 'ad2_name': 'HHV8_VIRF2',
 'fitness': np.float64(675.75586148167),
 'in_degree': 48,
 'out_degree': 0,
 'is_lo': True}

## 6. Analysing the landscape

Next, we’ll count local optima, measure ruggedness, examine interactions and assess the trend toward the best observed configuration.

In [9]:
from graphfla import analysis

### 6.1 Number of local optima

We can start with `landscape.n_lo`. A local optimum has no improving route out of its immediate neighbourhood, including neutral moves within a plateau. Each connected neutral optimum plateau counts once. Multiple local optima mean that successive improvements can lead to different endpoints.

In [10]:
print(f"Number of local optima: {landscape.n_lo}")

Number of local optima: 2


### 6.2 Roughness-to-slope ratio

For a view of ruggedness, we can calculate `r_s_ratio`. It compares deviations from an additive model with the average magnitude of its coefficients. Larger values indicate more residual roughness relative to the additive trend; a nearly zero fitted slope can give an infinite ratio.


Here, an additive model assigns a separate contribution to each domain choice in each position. Deviations reveal response variation that those independent contributions do not explain.

In [11]:
roughness_to_slope = analysis.r_s_ratio(landscape)
print(f"Roughness-to-slope ratio: {roughness_to_slope:.4f}")

Roughness-to-slope ratio: 0.7802


### 6.3 Interactions between domains

Replacing one domain may increase activation with one partner but decrease it with another. We can examine this context dependence using GraphFLA’s epistasis classification. The configurations are whole-domain combinations, so the changes concern domain replacements.

We can explore these patterns with `classify_epistasis()`, which classifies four-configuration graph motifs:

| Category | Interpretation |
|---|---|
| `magnitude` | Changes retain their direction of effect across contexts. |
| `sign` | One change reverses its direction of effect across contexts. |
| `reciprocal_sign` | Both changes reverse their direction of effect across contexts. |

The output gives their proportions among the detected motifs. The `positive` and `negative` fields form a separate partition.

In [12]:
interactions = analysis.classify_epistasis(landscape, seed=42)
interactions

EpistasisClassification(magnitude=0.5348888888888889, sign=0.348, reciprocal_sign=0.11711111111111111, positive=0.8136777777777778, negative=0.18632222222222217)

### 6.4 Fitness-distance correlation

Finally, we can use `fdc` to check whether higher-scoring configurations tend to lie closer to the best observed configuration—a simple view of navigability.


Distance counts how many of the two domain positions differ from the best construct. It does not measure amino-acid sequence similarity between domains.

We use Spearman correlation to compare ranks. A negative value means higher scores tend to occur nearer the optimum; a value near zero indicates little monotonic association. This trend does not guarantee an improving path from every configuration.

In [13]:
fitness_distance_r = analysis.fdc(landscape, method="spearman")
print(f"Fitness-distance correlation: {fitness_distance_r:.4f}")

 - Calculating distances to global optimum using mixed_distance...


   - Distances to GO calculated and added as node attribute 'dist_go'.


Fitness-distance correlation: -0.2621


## 7. Running several analyses together

Once we’ve tried the metrics individually, we can collect them with `analysis.profile()`. Here, we combine the ruggedness measure and fitness-distance correlation in a pandas Series. Setting `on_error="raise"` displays errors immediately.

In [14]:
analysis.profile(
    landscape,
    include=["r_s_ratio", "fdc"],
    on_error="raise",
    progress=False,
)

r_s_ratio    0.780195
fdc         -0.262059
dtype: float64

## 8. Analysis reference

For further exploration, we can choose from the **30 analysis functions** below. Each takes a built `landscape` as its first argument. In these functions, *fitness* refers to the outcome selected as `f`.

Walsh–Hadamard and higher-order regression are included for reference. Their cost grows rapidly with the number of variables and levels, so we leave them out of the worked examples.

| Group | Function | Result |
| --- | --- | --- |
| Fitness | `fitness_distribution` | Dictionary of distribution summaries, including skewness, kurtosis, CV and Cauchy location. |
| Fitness | `fitness_effect_distribution` | List of signed effects for one mutation across matched backgrounds. |
| Ruggedness | `local_optima_ratio` | Local-optimum count divided by configuration count. |
| Ruggedness | `r_s_ratio` | Additive-model residual roughness divided by mean absolute coefficient. |
| Ruggedness | `autocorrelation` | Fitness autocorrelation along sampled walks. |
| Ruggedness | `gradient_intensity` | Mean absolute change on improving edges divided by mean fitness. |
| Correlation | `fdc` | Fitness-distance correlation: Spearman or Pearson. |
| Correlation | `neighbor_fitness_correlation` | Correlation between fitness and mean neighbour fitness: Pearson, Spearman or Kendall. |
| Correlation | `basin_fitness_correlation` | Correlation between peak fitness and greedy basin size: Spearman or Pearson. |
| Correlation | `fitness_flattening_index` | Trend in successive fitness gains along greedy walks: Spearman or Pearson. |
| Navigability | `global_optima_accessibility` | Fraction of configurations with an improving route to the global optimum. |
| Navigability | `local_optima_accessibility` | DataFrame of accessibility values for specified local optima. |
| Navigability | `mean_distance_to_global_optimum` | Mean sequence/configuration distance to the global optimum. |
| Navigability | `mean_distance_to_local_optima` | DataFrame of mean sequence/configuration distances to specified local optima. |
| Navigability | `mean_path_length_to_global_optimum` | Mean shortest graph-path length, excluding unreachable starts. |
| Navigability | `mean_path_length_to_local_optima` | DataFrame of shortest-path summaries, excluding unreachable starts. `lo=None` selects the global optimum. |
| Robustness | `neutrality` | Fraction of neighbour pairs within the fitness-difference threshold. |
| Robustness | `evolvability_enhancing_fraction` | Fraction of all directed neighbour pairs classified as EE using non-focal neighbourhoods and BH-corrected tests. |
| Robustness | `single_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at one position; positive or negative tests. |
| Robustness | `all_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at all positions. |
| Epistasis | `idiosyncratic_index` | Normalized spread of one mutation’s effects across backgrounds. |
| Epistasis | `global_idiosyncratic_index` | Mean idiosyncratic index across eligible mutations. |
| Epistasis | `gamma` | Correlation of mutation effects across matched two-site backgrounds. |
| Epistasis | `gamma_star` | Consistency of mutation-effect signs across backgrounds. |
| Epistasis | `diminishing_returns_index` | Association of background fitness with mean beneficial gain: Pearson, Spearman or regression slope. |
| Epistasis | `increasing_costs_index` | Association of background fitness with mean deleterious cost: Pearson, Spearman or regression slope. |
| Epistasis | `higher_order_epistasis` | Per-order cumulative R² and incremental fit gains; accepts an existing W-H result without refitting. |
| Epistasis | `walsh_hadamard` | Coefficients and an order summary with fit gains and the fitted-model variance spectrum. |
| Epistasis | `classify_epistasis` | Object with magnitude, sign, reciprocal-sign, positive and negative epistasis proportions. |
| Epistasis | `extradimensional_bypass` | Object with bypass proportion, mean length and motif counts. |

### Data source

Giddins M. et al. (2025). [*Combinatorial protein engineering identifies potent CRISPR activators with reduced toxicity*. Nature Communications 16, 11114](https://doi.org/10.1038/s41467-025-65986-4).

The repository CSV contains the bipartite activation scores curated from the authors’ supplementary data.